In [2]:
import os, re, sys
import pickle
import numpy as np
import pandas as pd

from rdkit import Chem
from rdkit.Chem import PandasTools
from rdkit.Chem import AllChem, Draw
from rdkit.Chem import Descriptors
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit.Chem.MolStandardize.standardize import canonicalize_tautomer_smiles

Chem.SetDefaultPickleProperties(Chem.PropertyPickleOptions.AllProps)  # Keep properties when dumping pickle
PandasTools.RenderImagesInAllDataFrames(images=True)
from rdkit import RDLogger; RDLogger.DisableLog('rdApp.*')
from rdkit.Chem.AllChem import GetMorganFingerprintAsBitVect

from rdkit.Chem.Draw import IPythonConsole
IPythonConsole.ipython_useSVG = True
IPythonConsole.molSize = 600, 400
# IPythonConsole.drawOptions.addAtomIndices = True

dbdir = 'C:/Users/Idener/Sync/DATABASES'

In [ ]:
# Compile LOTUS database (approx. 40 min)
df_lotus = pd.read_csv(R'C:\Users\Idener\Sync\DATABASES\LOTUS\LOTUS_DB_2021.smi',
                       sep='\t', header=None, names=['SMILES', 'LTS_ID'])
PandasTools.AddMoleculeColumnToFrame(df_lotus, smilesCol='SMILES')
df_lotus['ISOSMILES'] = df_lotus['ROMol'].map(Chem.MolToSmiles)  # Better stick with RDKit Smiles convention
df_lotus['Formula'] = df_lotus['ROMol'].map(AllChem.CalcMolFormula)
df_lotus = df_lotus[['LTS_ID', 'ISOSMILES', 'Formula', 'ROMol']]

df_lotus.drop(columns=['ROMol']).to_csv('data/Database_LOTUS.csv', index=False, sep='\t')
df_lotus.to_pickle('data/Database_LOTUS.pkl')

In [ ]:
sys.path.append(R'C:\Users\Idener\MEGA\Algoritmi\drug_design_modules')
from map4 import MAP4Calculator; map4 = MAP4Calculator(2048)
import tmap as tm; enc = tm.Minhash(2048)

# Compile LIPID MAPS database
# lipid_mols = [x for x in Chem.SDMolSupplier(f"{dbdir}/LIPIDMAPS/LIPID_MAPS_Structure_Database_2024.sdf")]
# dict_list = [x.GetPropsAsDict() for x in lipid_mols]
# df_lipidmaps = pd.DataFrame(dict_list)
# df_lipidmaps.to_csv(f"{dbdir}/LIPIDMAPS/LIPID_MAPS_Structure_Database_2024.csv", index=False, sep='\t')

# Compile a reduced LIPID MAPS database
df_lipidmaps = pd.read_csv(f"{dbdir}/LIPIDMAPS/LIPID_MAPS_Structure_Database_2024.csv", sep='\t')
df_lipidmaps = df_lipidmaps.rename(columns={'LM_ID':'ID'}).dropna(subset='SMILES')
PandasTools.AddMoleculeColumnToFrame(df_lipidmaps, smilesCol='SMILES')
df_lipidmaps = df_lipidmaps.dropna(subset=['ROMol'])
df_lipidmaps['ISOSMILES'] = df_lipidmaps['ROMol'].map(Chem.MolToSmiles)
df_lipidmaps['Formula'] = df_lipidmaps['ROMol'].map(AllChem.CalcMolFormula)
df_lipidmaps['MW'] = df_lipidmaps['ISOSMILES'].apply(lambda x: Descriptors.MolWt(Chem.MolFromSmiles(x)))
df_lipidmaps['Ontology'] = df_lipidmaps[['MAIN_CLASS', 'SUB_CLASS']].apply(
                                lambda x: [i for i in x if pd.notna(i)], axis=1)
df_lipidmaps = df_lipidmaps[['ID', 'ISOSMILES', 'Formula', 'MW', 'Ontology', 'ROMol']]
df_lipidmaps.drop(columns='ROMol').to_csv(f"data/Database_lipidmaps.csv", index=False, sep='\t')
df_lipidmaps.to_pickle('data/Database_lipidmaps.pkl')
df_lipidmaps